# Full-text Model with Triplet Learning out-of-Corpus

This notebook perform evaluation of the full-text model using triplet learning in the out-of-corpus inference setting.

# Set up

In [3]:
pip install tensorflow

  Using cached wheel-0.46.3-py3-none-any.whl.metadata (2.4 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 281.8/281.8 MB 3.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.5/57.5 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.5/6.5 MB 45.3 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 38.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 26.0 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 32.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.1/5.1 MB 42.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.9/71.9 kB 4.1 MB/s eta 0:00:00
Using cached wheel-0.46.3-py3-none-any.whl (30 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.2/400.2 kB 22.4 MB/s eta 0:00:00

[notice] A new release of pip is available: 2

In [4]:
import numpy as np
import pandas as pd
import pickle

## Class definition

In [5]:
import tensorflow as tf

class FeedForward(tf.keras.layers.Layer):
  def __init__(self, d_model, dropout_rate=0.1):
    super().__init__()
    self.seq = tf.keras.Sequential([
      tf.keras.layers.Dense(d_model, activation='relu'),
      tf.keras.layers.Dense(d_model),
      tf.keras.layers.Dropout(dropout_rate)
    ])
    self.add = tf.keras.layers.Add()
    self.layer_norm = tf.keras.layers.LayerNormalization()

  def call(self, x):
    x = self.add([x, self.seq(x)])
    x = self.layer_norm(x)
    return x


class FinetuneEncoder(tf.keras.layers.Layer):
  def __init__(self, ff_layers, dropout_rate=0.1):
    super().__init__()
    self.num_blocks = ff_layers
    self.ff_layers = [FeedForward(d_model, dropout_rate) for d_model in self.num_blocks]
    self.embedding = tf.keras.layers.Lambda(lambda x: tf.math.l2_normalize(x, axis=1))

  def call(self, x):
    for layer in self.ff_layers:
      x = layer(x)
    x = self.embedding(x)
    return x


class DistanceLayer(tf.keras.layers.Layer):
  def __init__(self, **kwargs):
      super().__init__(**kwargs)

  def call(self, anchor, positive, negative):
      ap_distance = tf.reduce_sum(tf.square(anchor - positive), -1)
      an_distance = tf.reduce_sum(tf.square(anchor - negative), -1)
      return (ap_distance, an_distance)


class TripletModel(tf.keras.Model):
  def __init__(self, siamese_network, margin=0.5):
      super().__init__()
      self.siamese_network = siamese_network
      self.margin = margin
      self.loss_tracker = tf.keras.metrics.Mean(name="loss")

  def call(self, inputs):
      return self.siamese_network(inputs)

  def train_step(self, data):
      with tf.GradientTape() as tape:
          loss = self._compute_loss(data)
      gradients = tape.gradient(loss, self.siamese_network.trainable_weights)
      self.optimizer.apply_gradients(
          zip(gradients, self.siamese_network.trainable_weights)
      )
      self.loss_tracker.update_state(loss)
      return {"loss": self.loss_tracker.result()}

  def test_step(self, data):
      loss = self._compute_loss(data)
      self.loss_tracker.update_state(loss)
      return {"loss": self.loss_tracker.result()}

  def _compute_loss(self, data):
      ap_distance, an_distance = self.siamese_network(data)
      loss = ap_distance - an_distance
      loss = tf.maximum(loss + self.margin, 0.0)
      return loss

  @property
  def metrics(self):
      # We need to list our metrics here so the `reset_states()` can be
      # called automatically.
      return [self.loss_tracker]


from sklearn.metrics import f1_score

def f1(x,y,thres):
    yx = np.zeros(x.shape[0])
    yy = np.ones(y.shape[0])
    yhx = np.zeros(x.shape[0])
    yhx[x > thres] = 1
    yhy = np.zeros(y.shape[0])
    yhy[y > thres] = 1
    return f1_score(np.concatenate([yx,yy]), np.concatenate([yhx,yhy]))

In [6]:
def run_model(answers, gpt1, gpt2, test_answers, test_gpt1, test_gpt2, epochs=50, seeds=(1111,2222,3333,4444,5555,6666,7777,8888,9999,1010)):
  valid_accuracies = []
  valid_f1s = []
  test_accuracies = []
  test_f1s = []

  for seed in seeds:
    np.random.seed(seed)

    #train-test-validation split
    sampling_indices = np.random.uniform(0,1,answers.shape[0])

    train_answers = answers[sampling_indices < 0.9]
    train_gpt1 = gpt1[sampling_indices < 0.9]
    train_gpt2 = gpt2[sampling_indices < 0.9]

    valid_answers = answers[sampling_indices >= 0.9]
    valid_gpt1 = gpt1[sampling_indices >= 0.9]
    valid_gpt2 = gpt2[sampling_indices >= 0.9]

    #build model
    emb_shape = 768
    embedding = FinetuneEncoder([768]*3)
    anchor_input = tf.keras.layers.Input(name="anchor", shape=[emb_shape])
    positive_input = tf.keras.layers.Input(name="positive", shape=[emb_shape])
    negative_input = tf.keras.layers.Input(name="negative", shape=[emb_shape])
    distances = DistanceLayer()(
        embedding(anchor_input),
        embedding(positive_input),
        embedding(negative_input),
    )
    distance_network = tf.keras.Model(
        inputs=[anchor_input, positive_input, negative_input], outputs=distances
    )
    triplet_model = TripletModel(distance_network, margin=0.1)

    #training
    triplet_model.compile(optimizer=tf.keras.optimizers.Adam(0.00001), weighted_metrics=[])
    triplet_model.fit([train_gpt1, train_gpt2, train_answers],
                      epochs=epochs, batch_size=2048,
                      validation_data=[valid_gpt1, valid_gpt2, valid_answers])

    #valid accuracy
    valid_emb_answers = embedding(valid_answers)
    valid_emb_gpt1 = embedding(valid_gpt1)
    valid_emb_gpt2 = embedding(valid_gpt2)
    valid_gpt_distances = ((valid_emb_gpt1 - valid_emb_gpt2)**2).numpy().sum(axis=1)
    valid_answer_distances = ((valid_emb_gpt1 - valid_emb_answers)**2).numpy().sum(axis=1)
    valid_accuracies.append((valid_gpt_distances < valid_answer_distances).mean())

    #tune threshold
    thresholds = []
    perf = []
    for thres in np.arange(0,1,0.01):
      thresholds.append(thres)
      perf.append(f1(valid_gpt_distances, valid_answer_distances, thres))

    triplet_thres = thresholds[perf.index(max(perf))]

    #valid f1
    valid_f1s.append(max(perf))

    #test accuracy
    test_emb_answers = embedding(test_answers)
    test_emb_gpt1 = embedding(test_gpt1)
    test_emb_gpt2 = embedding(test_gpt2)
    test_gpt_distances = ((test_emb_gpt1 - test_emb_gpt2)**2).numpy().sum(axis=1)
    test_answer_distances = ((test_emb_gpt1 - test_emb_answers)**2).numpy().sum(axis=1)
    test_accuracies.append((test_gpt_distances < test_answer_distances).mean())

    #test f1
    test_f1s.append(f1(test_gpt_distances, test_answer_distances, triplet_thres))
  return valid_accuracies, test_accuracies, valid_f1s, test_f1s

## NQ Data - SQUAD test

In [8]:
train_data = 'NQ_GPT4'
test_data = 'SQUAD_GPT4'

with open(f'/home/aanis/Metric-Models-for-Detection-of-LLM-Texts/Data/{train_data}_answer_embs.obj', 'rb') as f:
  answers = pickle.load(f)
with open(f'/home/aanis/Metric-Models-for-Detection-of-LLM-Texts/Data/{train_data}_gpt1_embs.obj', 'rb') as f:
  gpt1 = pickle.load(f)
with open(f'/home/aanis/Metric-Models-for-Detection-of-LLM-Texts/Data/{train_data}_gpt2_embs.obj', 'rb') as f:
  gpt2 = pickle.load(f)

with open(f'/home/aanis/Metric-Models-for-Detection-of-LLM-Texts/Data/{test_data}_answer_embs.obj', 'rb') as f:
  test_answers = pickle.load(f)
with open(f'/home/aanis/Metric-Models-for-Detection-of-LLM-Texts/Data/{test_data}_answer_embs.obj', 'rb') as f:
  test_gpt1 = pickle.load(f)
with open(f'/home/aanis/Metric-Models-for-Detection-of-LLM-Texts/Data/{test_data}_answer_embs.obj', 'rb') as f:
  test_gpt2 = pickle.load(f)

FileNotFoundError: [Errno 2] No such file or directory: '/home/aanis/Metric-Models-for-Detection-of-LLM-Texts/Data/NQ_GPT4_answer_embs.obj'

In [ ]:
valid_accuracies, test_accuracies, valid_f1s, test_f1s = run_model(answers, gpt1, gpt2, test_answers, test_gpt1, test_gpt2)

In [ ]:
np.mean(valid_accuracies), np.mean(test_accuracies), np.mean(valid_f1s), np.mean(test_f1s)